## 전처리

#### 1. 데이터 불러오기

In [1]:
# 데이터 불러오기

import pandas as pd

df = pd.read_csv("news_dataset_economy.csv")

# 데이터 백업
df_raw = df.copy()

print("원본 데이터 개수:", len(df))

원본 데이터 개수: 1000


#### 2. 중복 기사 제거

In [2]:
# 제목과 내용으로 중복 기사 확인
df[df.duplicated(subset=['title','content'], keep=False)]

,title,content,date,press,url
11,나프타 수출 금지·요소 매점매석 금지,[앵커]\n국내 나프타 재고가 2주치 정도에 불과한 상황에 이르자 정부는 오늘 밤인...,2026-03-26 22:11:00,YTN,https://n.news.naver.com/mnews/article/052/000...
12,"[속보] 2차 최고가격 '휘발유 1,934원·경유 1,923원'...주유소 기름값 ...",주유소 기름값 2천 원대 초반 예상\n주유소 전국 평균 휘발윳값…최고가격보다 100...,2026-03-26 22:11:00,YTN,https://n.news.naver.com/mnews/article/052/000...
20,“누구나 편하게 즐겨요”…청남대 모노레일 개통,"[KBS 청주] [앵커]\n옛 대통령 별장, 청남대에 모노레일이 개통됐습니다.\n충...",2026-03-26 22:02:00,KBS,https://n.news.naver.com/mnews/article/056/001...
32,종량제 봉투 충분하다지만…사재기 현상에 공급 불안,[KBS 청주] [앵커]\n중통 사태의 여파는 시민들의 일상 생활 곳곳에 미치고 있...,2026-03-26 21:59:00,KBS,https://n.news.naver.com/mnews/article/056/001...
40,‘교복 지원 조례 개정안’ 상임위 통과…‘바우처’ 도입,"[KBS 춘천]강원도의회 교육위원회가 오늘(26일), '학교 교복 지원 조례 일부 ...",2026-03-26 21:55:00,KBS,https://n.news.naver.com/mnews/article/056/001...
46,"대구시, 승용차 5부제 등 특별 대책 추진",[KBS 대구]중동 사태와 관련해 대구시가 특별 대책을 추진합니다.\n우선 공공부문...,2026-03-26 21:49:00,KBS,https://n.news.naver.com/mnews/article/056/001...
50,부실채권에 연체율 느는데…지주회장은 ‘연봉킹’,"[KBS 전주][앵커]\nJB금융지주가 외형 성장을 이어가고 있지만, 은행 연체율이...",2026-03-26 21:45:00,KBS,https://n.news.naver.com/mnews/article/056/001...
52,BNK 빈 회장 체제 3년 더…“지역금융 강화”,[KBS 부산] [앵커]\nBNK금융이 역대 최고 실적을 내 주가가 고공행진 중입니...,2026-03-26 21:43:00,KBS,https://n.news.naver.com/mnews/article/056/001...
58,일할수록 손해…특수노동자 ‘고유가’에 휘청,[KBS 부산] [앵커]\n중동발 고유가로 차량 5부제가 시행되는 등 정부가 에너...,2026-03-26 21:38:00,KBS,https://n.news.naver.com/mnews/article/056/001...
83,나프타 수출 금지·요소 매점매석 금지,[앵커]\n국내 나프타 재고가 2주치 정도에 불과한 상황에 이르자 정부는 오늘 밤인...,2026-03-26 21:15:00,YTN,https://n.news.naver.com/mnews/article/052/000...


In [3]:
# 중복 기사 제거 (같은 기사 중 최신 기사만 유지)
# 1.날짜 형식 변환
df['date'] = pd.to_datetime(df['date'])

# 2.title + content 기준 중복 제거
df = df.sort_values("date").drop_duplicates(subset=['title','content'], keep="last")

# 백업
df_deduplicated = df.copy()

print("중복 제거 후:", len(df))

중복 제거 후: 978


In [4]:
# URL로 중복 기사 확인
df[df.duplicated(subset=['url'], keep=False)]

,title,content,date,press,url


In [5]:
# 중복 기사 제거 (같은 기사 중 최신 기사만 유지)

# url 기준 중복 제거
df = df.sort_values("date").drop_duplicates(subset=['url'], keep="last")

# 백업
df_deduplicated_2 = df.copy()

print("중복 제거 후:", len(df))

중복 제거 후: 978


#### 3. 언론사별 기사 개수 확인 및 기사가 13개 이상인 언론사만 필터링

In [6]:
# 언론사별 기사 개수 확인
df['press'].value_counts()

press
뉴스1        72
이데일리       71
파이낸셜뉴스     67
한국경제       50
매일경제       50
           ..
강원도민일보      1
스포츠조선       1
주간경향        1
kbc광주방송     1
OSEN        1
Name: count, Length: 66, dtype: int64

In [7]:
# 기사 13개 이상인 언론사만 필터링

press_counts = df['press'].value_counts()

valid_press = press_counts[press_counts >= 13].index

df = df[df['press'].isin(valid_press)]

# 백업
df_press_filtered = df.copy()

print("언론사 필터링 후:", len(df))
print("남은 언론사 수:", df['press'].nunique())


언론사 필터링 후: 760
남은 언론사 수: 23


#### 4. 제목과 본문 결합

In [8]:
# 제목 + 본문 결합

df['text'] = df['title'].fillna('') + " " + df['content'].fillna('')

# 백업
df_text_combined = df.copy()

#### 5. 텍스트 정제

In [9]:
# 텍스트 정제(특수문자 / HTML / 이메일 / URL 제거)
import re

def clean_text(text):

    text = re.sub(r'<.*?>', ' ', text)           # html 태그 제거
    text = re.sub(r'http\S+|www\S+', ' ', text)  # url 제거
    text = re.sub(r'\S+@\S+', ' ', text)         # 이메일 제거
    text = re.sub(r'[^가-힣\s]', ' ', text)      # 특수문자 제거
    text = re.sub(r'\s+', ' ', text)             # 공백 정리

    return text.strip()


df['clean_text'] = df['text'].apply(clean_text)

# 백업
df_text_cleaned = df.copy()

In [8]:
!pip install kiwipiepy


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### 6. 형태소 분석(명사+동사+형용사)

In [10]:
# 형태소 분석(명사 + 동사 + 형용사)
from kiwipiepy import Kiwi

kiwi = Kiwi()
target_pos = ["NNG", "NNP", "VV", "VA"]

def tokenize(text):

    tokens = []

    result = kiwi.analyze(text)[0][0]

    for token, pos, _, _ in result:
        if pos in target_pos:
            tokens.append(token)

    return tokens

df["tokens"] = df["clean_text"].apply(tokenize)

df_tokenized = df.copy()

#### 7. 불용어 제거

In [11]:
# 불용어 제거

stopwords = ['기자','사진','뉴스','연합뉴스','뉴시스','보도','관련','이번','대한','통해', '하다', '이다',
'지난','현재','이날','오전','오후','당시','이후','앞서','최근','정부','관계자','업계','측','대통령','국회','위원회',
'등','것','수','때','위해','경우','때문','대해','지난해','올해','내년', '년','월','일','억','만','천''파이낸셜뉴스',
'KBS','한국경제','매일경제','이데일리','SBS Biz','부산일보','연합뉴스','서울경제','머니투데이','세계일보','국제신문'
]

def remove_stopwords(tokens):
    
    return [t for t in tokens if t not in stopwords]

df['tokens'] = df['tokens'].apply(remove_stopwords)

df_stopwords_removed = df.copy()


In [12]:
df = df_stopwords_removed.copy()

In [13]:
# 추가적인 불용어 제거

stopwords = [
'가능','상황','수준','계획','내용','포함','실제','방안','검토','일각','입장','경우','관련','통해',

# 뉴스 문체 단어
'발표','조사','응답','보고','설명','강조','언급','진단','요청','건의',

# 일반 서술 단어
'전체','가운데','이전','이후','처음','주요','전반','핵심','본격','적극','철저',

# 추상적인 단어
'효과','역할','문제','요소','체계','기반','구성','구축','확보','강화','지원',

# 시간 표현
'작년','새해','연내','시기','기간','차례',

# 분석에 의미 약한 동사
'나타나','밝히','따르','보이','점치','내다보','늘리','줄이','높이','갖추','만들',

# 문맥 보조 단어
'대상','이용','통하','위하','대하'
]

df['tokens'] = df['tokens'].apply(remove_stopwords)
df_stopwords_removed2 = df.copy()

#### 8. 한 글자 토큰 및 숫자 토큰 제거

In [14]:
# 한 글자인 토큰 제거

def remove_single_char(tokens):

    return [t for t in tokens if len(t) > 1]


df['tokens'] = df['tokens'].apply(remove_single_char)

# 백업
df_single_removed = df.copy()

# 숫자 토큰 제거

def remove_numbers(tokens):

    return [t for t in tokens if not t.isdigit()]


df['tokens'] = df['tokens'].apply(remove_numbers)

# 최종 전처리 백업
df_preprocessed = df.copy()

print("최종 데이터 개수:", len(df_preprocessed))


최종 데이터 개수: 760


#### 9. 최종 전처리 데이터 저장

In [15]:
# 최종 전처리 데이터 저장

df_preprocessed.to_csv("news_preprocessed_economy.csv", index=False)

#### 10. 최종 데이터 확인

In [16]:
# 데이터 확인
df = df.reset_index(drop=True)
df

,title,content,date,press,url,text,clean_text,tokens
0,"몽키트래블, 태국 호텔 예약 혜택 확대",동남아 전문 여행사 몽키트래블이 태국 호텔 예약 서비스에서 전용 혜택을 확대했다고 ...,2026-03-26 17:41:00,머니투데이,https://n.news.naver.com/mnews/article/008/000...,"몽키트래블, 태국 호텔 예약 혜택 확대 동남아 전문 여행사 몽키트래블이 태국 호텔...",몽키트래블 태국 호텔 예약 혜택 확대 동남아 전문 여행사 몽키트래블이 태국 호텔 예...,"[몽키트래블, 태국, 호텔, 예약, 혜택, 확대, 동남아, 전문, 여행사, 몽키트래..."
1,"일가족 사망 비극에…복지부, 경찰·교수 만나 '자살 시도자' 대응 논의",최근 일가족이 자살로 사망하는 사건이 발생한 가운데 보건복지부가 26일 오전 한국생...,2026-03-26 17:41:00,머니투데이,https://n.news.naver.com/mnews/article/008/000...,"일가족 사망 비극에…복지부, 경찰·교수 만나 '자살 시도자' 대응 논의 최근 일가족...",일가족 사망 비극에 복지부 경찰 교수 만나 자살 시도자 대응 논의 최근 일가족이 자...,"[일가족, 사망, 비극, 복지부, 경찰, 교수, 만나, 자살, 시도, 대응, 논의,..."
2,"국민성장펀드, 리벨리온에 2500억 첫 직접 투자","""K엔비디아 육성할 것""\nAI 반도체 스타트업 지원\n국민성장펀드가 이른바 'K엔...",2026-03-26 17:41:00,매일경제,https://n.news.naver.com/mnews/article/009/000...,"국민성장펀드, 리벨리온에 2500억 첫 직접 투자 ""K엔비디아 육성할 것""\nAI ...",국민성장펀드 리벨리온에 억 첫 직접 투자 엔비디아 육성할 것 반도체 스타트업 지원 ...,"[국민, 성장, 펀드, 직접, 투자, 엔비디아, 육성, 반도체, 스타트업, 국민, ..."
3,"LS에코에너지·호주 라이너스, 300억 CB 교환 '희토류 동맹'",LS에코에너지는 글로벌 희토류 원료 공급 2위 기업인 호주 라이너스와 상호 투자에 ...,2026-03-26 17:41:00,한국경제,https://n.news.naver.com/mnews/article/015/000...,"LS에코에너지·호주 라이너스, 300억 CB 교환 '희토류 동맹' LS에코에너지는 ...",에코에너지 호주 라이너스 억 교환 희토류 동맹 에코에너지는 글로벌 희토류 원료 공급...,"[에코, 에너지, 호주, 라이너스, 교환, 희토, 동맹, 에코, 에너지, 글로벌, ..."
4,한화생명 판매법인 분리 5년…연봉 1억 넘는 설계사 5606명,한화생명금융서비스가 출범 5년 만에 매출 2조4379억원을 기록했다.\n26일 한화...,2026-03-26 17:41:00,매일경제,https://n.news.naver.com/mnews/article/009/000...,한화생명 판매법인 분리 5년…연봉 1억 넘는 설계사 5606명 한화생명금융서비스가 ...,한화생명 판매법인 분리 년 연봉 억 넘는 설계사 명 한화생명금융서비스가 출범 년 만...,"[한화생명, 판매, 법인, 분리, 연봉, 설계사, 한화생명, 금융, 서비스, 출범,..."
...,...,...,...,...,...,...,...,...
755,"목돈부담 큰 계약금 마련 지원…코인베이스, 코인담보 모기지 출시","코인베이스, ‘페니메이 승인 모기지‘ 베터홈앤드파이낸스와 협업\n비트코인·USDC ...",2026-03-26 22:16:00,이데일리,https://n.news.naver.com/mnews/article/018/000...,"목돈부담 큰 계약금 마련 지원…코인베이스, 코인담보 모기지 출시 코인베이스, ‘페니...",목돈부담 큰 계약금 마련 지원 코인베이스 코인담보 모기지 출시 코인베이스 페니메이 ...,"[목돈, 부담, 계약금, 마련, 코인, 베이스, 코인, 담보, 모기지, 출시, 코인..."
756,정유업계 “2차 최고가격제 적극 협조…가격·수급 안정 총력”,대한석유협회와 국내 정유업계는 26일 정부가 발표한 비상 경제 대응 방안에 따라 가...,2026-03-26 22:16:00,조선비즈,https://n.news.naver.com/mnews/article/366/000...,정유업계 “2차 최고가격제 적극 협조…가격·수급 안정 총력” 대한석유협회와 국내 정...,정유업계 차 최고가격제 적극 협조 가격 수급 안정 총력 대한석유협회와 국내 정유업계...,"[정유업, 최고, 가격, 협조, 가격, 수급, 안정, 총력, 석유, 협회, 국내, ..."
757,한-브라질 외교장관회담…조현 '對韓 원유수출 확대' 협력제안,(서울=연합뉴스) 민선희 기자 = 조현 외교부 장관은 26일(현지시간) 프랑스에서 ...,2026-03-26 22:17:00,연합뉴스,https://n.news.naver.com/mnews/article/001/001...,한-브라질 외교장관회담…조현 '對韓 원유수출 확대' 협력제안 (서울=연합뉴스) 민선...,한 브라질 외교장관회담 조현 원유수출 확대 협력제안 서울 연합뉴스 민선희 기자 조현...,"[브라질, 외교, 장관, 회담, 조현, 원유, 수출, 확대, 협력, 제안, 서울, ..."
758,"정유업계, 2차 최고가격제 협조…""석유제품 국내 우선 공급""","정부, 2차 석유 최고가격제 27일 0시부터 시행\n(서울=뉴스1) 원태성 기자 =...",2026-03-26 22:18:00,뉴스1,https://n.news.naver.com/mnews/article/421/000...,"정유업계, 2차 최고가격제 협조…""석유제품 국내 우선 공급"" 정부, 2차 석유 최고...",정유업계 차 최고가격제 협조 석유제품 국내 우선 공급 정부 차 석유 최고가격제 일 ...,"[정유업, 최고, 가격, 협조, 석유, 제품, 국내, 우선, 공급, 석유, 최고, ..."
